# Notebook 06 — Integração dos sentimentos à previsão da inadimplência

Este notebook integra as variáveis de sentimento selecionadas no Notebook 05 aos três
modelos-base do Notebook 02 (**ARIMAX**, **XGBoost** e **SVR**), testando individualmente
cada lag L1–L6 de cada modelo de sentimento.

### Estrutura do notebook

| Seção | Conteúdo |
|-------|----------|
| 2–7   | Importações, configurações, dados e split |
| 8     | Funções auxiliares (compartilhadas entre algoritmos) |
| 9     | ARIMAX — todos os lags × todos os sentimentos |
| 10    | XGBoost — todos os lags × todos os sentimentos |
| 11    | SVR — todos os lags × todos os sentimentos |
| 12    | Resumo e exportação para o Notebook 07 |

### Benchmarks do Notebook 02

| Modelo | MAE | RMSE | R² |
|--------|-----|------|-----|
| ARIMA(2,1,4) + bias | 0.143668 | 0.184052 | 0.687889 |
| SVR(k=2) + bias | 0.255929 | 0.283914 | 0.257322 |
| XGBoost(k=6) + bias | 0.278016 | 0.317100 | 0.073554 |

> Os baselines de SVR e XGBoost são recomputados localmente neste notebook
> sobre `base_completa.csv` para garantir comparabilidade direta com as
> versões com sentimento.


## 2. Importações

In [1]:
import warnings
warnings.filterwarnings('ignore')

import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

from statsmodels.tsa.statespace.sarimax import SARIMAX
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline
from sklearn.svm import SVR
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from xgboost import XGBRegressor

print("Bibliotecas importadas com sucesso.")


Bibliotecas importadas com sucesso.


## 3. Configurações centrais

In [2]:
ARQUIVO_BASE  = "base_completa.csv"
VARIAVEL_ALVO = "inad_total_tplus3"
H             = 3
SPLIT_DATE    = "2024-01-01"
ORDEM_ARIMA   = (2, 1, 4)
K_SVR         = 2
K_XGB         = 6
MIN_OBS_TREINO = 20
MIN_OBS_TESTE  = 5
COLOR          = "#0e5764"

# Benchmarks Notebook 02 (ARIMA)
RMSE_NB02_ARIMA = 0.184052
MAE_NB02_ARIMA  = 0.143668
R2_NB02_ARIMA   = 0.687889

# Modelos de sentimento selecionados no Notebook 05
MODELOS_SENTIMENTO = {
    'nltk_copom'    : {'tipo': 'copom',        'modelo': 'nltk',    'label': 'NLTK/VADER Copom'},
    'mistral_copom' : {'tipo': 'copom',        'modelo': 'mistral', 'label': 'Mistral Copom'},
    'media_copom'   : {'tipo': 'copom',        'modelo': 'media',   'label': 'Media dos Modelos Copom'},
    'finbert_estat' : {'tipo': 'estatisticas', 'modelo': 'finbert', 'label': 'FinBERT-PT-BR Estat.'},
}

LAGS = list(range(1, 7))   # L1 a L6 — testados individualmente

print("=== Configurações ===")
print(f"Arquivo        : {ARQUIVO_BASE}")
print(f"Variável-alvo  : {VARIAVEL_ALVO}")
print(f"Split          : {SPLIT_DATE}")
print(f"ARIMA          : {ORDEM_ARIMA}")
print(f"SVR k={K_SVR}  |  XGBoost k={K_XGB}")
print(f"Lags testados  : L1 a L6 (individualmente)")


=== Configurações ===
Arquivo        : base_completa.csv
Variável-alvo  : inad_total_tplus3
Split          : 2024-01-01
ARIMA          : (2, 1, 4)
SVR k=2  |  XGBoost k=6
Lags testados  : L1 a L6 (individualmente)


## 4. Carregamento e validação da base

In [3]:
base = pd.read_csv(ARQUIVO_BASE)
base.columns = [c.strip().lower() for c in base.columns]
base = base.drop(columns=["unnamed: 0"], errors="ignore")

base["data"]      = pd.to_datetime(base["data"],      errors="coerce")
base["data_alvo"] = pd.to_datetime(base["data_alvo"], errors="coerce")
base = base.sort_values("data").reset_index(drop=True)

obrig = ["data", "data_alvo", VARIAVEL_ALVO]
faltando = [c for c in obrig if c not in base.columns]
if faltando:
    raise ValueError(f"Colunas ausentes: {faltando}")

print(f"Base: {base.shape[0]} linhas × {base.shape[1]} colunas")
print(f"Período data      : {base['data'].min().date()} → {base['data'].max().date()}")
print(f"Período data_alvo : {base['data_alvo'].min().date()} → {base['data_alvo'].max().date()}")
print(f"Nulos em {VARIAVEL_ALVO}: {base[VARIAVEL_ALVO].isna().sum()}")


Base: 75 linhas × 82 colunas
Período data      : 2019-07-01 → 2025-09-01
Período data_alvo : 2019-10-01 → 2025-12-01
Nulos em inad_total_tplus3: 0


## 5. Mapeamento das colunas de sentimento (L1–L6)

In [4]:
def buscar_coluna(base, tipo, modelo, lag):
    candidatas = [
        c for c in base.columns
        if tipo.lower() in c and modelo.lower() in c and f"_l{lag}" in c
    ]
    return candidatas[0] if candidatas else None

# Mapear todos os lags para cada modelo de sentimento
mapa_cols = {}
for chave, cfg in MODELOS_SENTIMENTO.items():
    mapa_cols[chave] = {}
    for lag in LAGS:
        col = buscar_coluna(base, cfg["tipo"], cfg["modelo"], lag)
        mapa_cols[chave][lag] = col

# Exibir disponibilidade
print("=== Disponibilidade de colunas por modelo × lag ===")
header = f"{'Modelo':<25}" + "".join(f"  L{l}" for l in LAGS)
print(header)
for chave, cfg in MODELOS_SENTIMENTO.items():
    row = f"{cfg['label']:<25}"
    for lag in LAGS:
        row += "  " + ("OK " if mapa_cols[chave][lag] else "---")
    print(row)


=== Disponibilidade de colunas por modelo × lag ===
Modelo                     L1  L2  L3  L4  L5  L6
NLTK/VADER Copom           OK   OK   OK   OK   OK   OK 
Mistral Copom              OK   OK   OK   OK   OK   OK 
Media dos Modelos Copom    OK   OK   OK   OK   OK   OK 
FinBERT-PT-BR Estat.       OK   OK   OK   OK   OK   OK 


## 6. Split treino / teste

In [5]:
split_dt    = pd.Timestamp(SPLIT_DATE)
base_fit    = base.dropna(subset=[VARIAVEL_ALVO, "data_alvo"]).copy()
base_treino = base_fit[base_fit["data_alvo"] <  split_dt].reset_index(drop=True)
base_teste  = base_fit[base_fit["data_alvo"] >= split_dt].reset_index(drop=True)

print(f"Treino: {base_treino['data_alvo'].min().date()} → "
      f"{base_treino['data_alvo'].max().date()}  | {len(base_treino)} obs.")
print(f"Teste : {base_teste['data_alvo'].min().date()} → "
      f"{base_teste['data_alvo'].max().date()}  | {len(base_teste)} obs.")
assert not (set(base_treino["data_alvo"]) & set(base_teste["data_alvo"])), "Sobreposição!"
print("Split OK — sem sobreposição.")


Treino: 2019-10-01 → 2023-12-01  | 51 obs.
Teste : 2024-01-01 → 2025-12-01  | 24 obs.
Split OK — sem sobreposição.


## 7. Funções auxiliares

In [6]:
# ── Métricas ─────────────────────────────────────────────────────────────────
def calcular_metricas(y_obs, y_pred, nome=""):
    y_obs  = np.array(y_obs,  dtype=float)
    y_pred = np.array(y_pred, dtype=float)
    mask   = ~(np.isnan(y_obs) | np.isnan(y_pred))
    y_obs, y_pred = y_obs[mask], y_pred[mask]
    if len(y_obs) == 0:
        return {"Modelo": nome, "MAE": np.nan, "RMSE": np.nan, "R2": np.nan,
                "Bias": np.nan, "N": 0}
    mae  = mean_absolute_error(y_obs, y_pred)
    rmse = float(np.sqrt(mean_squared_error(y_obs, y_pred)))
    r2   = r2_score(y_obs, y_pred)
    bias = float(np.mean(y_obs - y_pred))
    lr   = LinearRegression().fit(y_obs.reshape(-1, 1), y_pred)
    return {"Modelo": nome, "MAE": round(mae, 6), "RMSE": round(rmse, 6),
            "R2": round(r2, 6), "Bias": round(bias, 6),
            "Slope": round(float(lr.coef_[0]), 6),
            "Intercept": round(float(lr.intercept_), 6), "N": len(y_obs)}

print("calcular_metricas definida.")


calcular_metricas definida.


In [7]:
# ── Walk-forward ARIMAX com bias correction ─────────────────────────────────
def walk_forward_arimax(endog, exog=None, train_size=30,
                        order=(2, 1, 4), h=3, padronizar=True):
    """
    Executa walk-forward para ARIMA/ARIMAX com horizonte h.

    A lógica replica o Notebook 02:
    - em cada iteração, o modelo é treinado apenas com o histórico disponível;
    - são previstos h passos à frente;
    - apenas o h-ésimo passo é guardado.

    Quando exog é informado, o modelo funciona como ARIMAX.
    Quando exog é None, o modelo funciona como ARIMA.
    """
    endog = endog.astype(float).dropna()

    if exog is not None and not exog.empty:
        dados = pd.concat([endog.rename("y"), exog], axis=1).dropna()
        endog = dados["y"].astype(float)
        exog  = dados.drop(columns=["y"]).astype(float)
    else:
        exog = None

    previsoes, idxs = [], []

    if len(endog) < train_size + h:
        return pd.Series(dtype=float)

    for i in range(train_size, len(endog) - h + 1):
        historico_y = endog.iloc[:i]
        historico_x = None
        futuro_x    = None

        if exog is not None:
            historico_x_raw = exog.iloc[:i]
            futuro_x_raw    = exog.iloc[i:i+h]

            if len(futuro_x_raw) < h:
                continue

            if padronizar:
                scaler = StandardScaler()
                historico_x = pd.DataFrame(
                    scaler.fit_transform(historico_x_raw),
                    index=historico_x_raw.index,
                    columns=historico_x_raw.columns
                )
                futuro_x = pd.DataFrame(
                    scaler.transform(futuro_x_raw),
                    index=futuro_x_raw.index,
                    columns=futuro_x_raw.columns
                )
            else:
                historico_x = historico_x_raw.copy()
                futuro_x    = futuro_x_raw.copy()

        try:
            ajuste = SARIMAX(
                historico_y,
                exog=historico_x,
                order=order,
                enforce_stationarity=True,
                enforce_invertibility=True
            ).fit(disp=False)

            forecast = ajuste.get_forecast(steps=h, exog=futuro_x)
            yhat = float(np.asarray(forecast.predicted_mean)[-1])

        except Exception:
            yhat = np.nan

        previsoes.append(yhat)
        idxs.append(endog.index[i + h - 1])

    return pd.Series(previsoes, index=idxs).dropna()


def estimar_arimax(nome, base_treino, base_teste, variavel_alvo,
                   cols_exog, ordem, h=3, min_obs=30, min_teste=5,
                   padronizar=True):
    """
    Estima ARIMA/ARIMAX por walk-forward h=3 e aplica bias correction.

    A correção de bias segue a lógica do Notebook 02:
    1. gera previsões walk-forward dentro do treino;
    2. calcula bias = média(observado_treino - previsto_treino);
    3. soma esse bias às previsões do teste;
    4. calcula métricas sem bias e com bias.
    """
    colunas = ["data_alvo", variavel_alvo] + list(cols_exog)

    base_modelo = (
        pd.concat([base_treino, base_teste], ignore_index=True)
        .assign(data_alvo=lambda d: pd.to_datetime(d["data_alvo"]))
        .sort_values("data_alvo")
        .dropna(subset=colunas)
        .drop_duplicates(subset="data_alvo", keep="last")
        .set_index("data_alvo")
    )

    y = base_modelo[variavel_alvo].astype(float)
    X = base_modelo[list(cols_exog)].astype(float) if cols_exog else None

    datas_teste = pd.DatetimeIndex(
        pd.to_datetime(base_teste["data_alvo"].dropna().unique())
    ).sort_values()

    if len(datas_teste) < min_teste:
        return None

    pos = np.where(y.index >= datas_teste.min())[0]
    if len(pos) == 0:
        return None

    train_size = int(pos[0]) - h + 1
    if train_size < min_obs:
        return None

    # 1. Bias estimado exclusivamente no treino
    y_treino = y[y.index < datas_teste.min()]
    X_treino = X.loc[y_treino.index] if X is not None else None

    min_obs_bias = min(15, max(5, len(y_treino) // 2))

    forecast_treino_bias = walk_forward_arimax(
        endog=y_treino,
        exog=X_treino,
        train_size=min_obs_bias,
        order=ordem,
        h=h,
        padronizar=padronizar
    )

    if len(forecast_treino_bias) > 0:
        y_true_bias = y_treino.loc[forecast_treino_bias.index]
        bias_estimado = float((y_true_bias - forecast_treino_bias).mean())
    else:
        bias_estimado = 0.0

    # 2. Walk-forward no teste oficial
    forecast_teste = walk_forward_arimax(
        endog=y,
        exog=X,
        train_size=train_size,
        order=ordem,
        h=h,
        padronizar=padronizar
    )

    forecast_teste = forecast_teste[forecast_teste.index.isin(datas_teste)]

    if len(forecast_teste) < min_teste:
        return None

    y_obs = y.loc[forecast_teste.index]
    y_pred_sem_bias = forecast_teste
    y_pred_com_bias = forecast_teste + bias_estimado

    metricas_sem_bias = calcular_metricas(
        y_obs,
        y_pred_sem_bias,
        nome=f"{nome}"
    )

    metricas_com_bias = calcular_metricas(
        y_obs,
        y_pred_com_bias,
        nome=f"{nome} + bias"
    )

    resultado = metricas_com_bias.copy()
    resultado.update({
        "nome_base": nome,
        "bias_estimado": round(bias_estimado, 6),
        "metricas_sem_bias": metricas_sem_bias,
        "metricas_com_bias": metricas_com_bias,
        "y_obs": y_obs.tolist(),
        "y_pred_sem_bias": y_pred_sem_bias.tolist(),
        "y_pred_com_bias": y_pred_com_bias.tolist(),
        "y_pred": y_pred_com_bias.tolist(),
        "datas": [str(d.date()) for d in forecast_teste.index],
    })

    return resultado

print("Funções ARIMAX com bias correction definidas.")


Funções ARIMAX com bias correction definidas.


In [8]:
# ── Walk-forward tabular com bias correction (XGBoost e SVR) ────────────────
def walk_forward_tabular(df, feature_cols, target_col, model_fn,
                         start_date=None, min_train=15):
    """
    Executa walk-forward para modelos tabulares, como XGBoost e SVR.

    A lógica preserva o horizonte h=3 por meio das colunas mes e data_alvo:
    - mes = mês de referência das variáveis disponíveis;
    - data_alvo = mês previsto, equivalente a mes + 3 meses.

    Para cada linha prevista, o treino usa apenas observações cuja data_alvo
    já estaria conhecida no mês de referência da previsão.
    """
    preds, idxs = [], []

    df = df.copy()
    df["mes"]       = pd.to_datetime(df["mes"])
    df["data_alvo"] = pd.to_datetime(df["data_alvo"])
    df = df.sort_values(["data_alvo", "mes"]).reset_index(drop=True)

    for i, row in df.iterrows():
        if start_date is not None and row["data_alvo"] < start_date:
            continue

        treino = df[df["data_alvo"] <= row["mes"]]

        if len(treino) < min_train:
            continue

        X_treino = treino[feature_cols]
        y_treino = treino[target_col]
        X_pred   = df.loc[[i], feature_cols]

        if X_treino.isna().any().any() or y_treino.isna().any() or X_pred.isna().any().any():
            continue

        try:
            modelo = model_fn()
            modelo.fit(X_treino, y_treino)
            yhat = float(modelo.predict(X_pred)[0])
        except Exception:
            yhat = np.nan

        preds.append(yhat)
        idxs.append(i)

    return pd.Series(preds, index=idxs).dropna()


def estimar_tabular(nome, model_fn, df_base, feature_cols,
                    target_col, split_dt, min_obs=15):
    """
    Estima XGBoost/SVR por walk-forward e aplica bias correction.

    A correção de bias segue o Notebook 02:
    1. gera previsões walk-forward dentro do treino;
    2. calcula bias = média(observado_treino - previsto_treino);
    3. soma esse bias às previsões do teste;
    4. calcula métricas sem bias e com bias.
    """
    mask_treino = df_base["data_alvo"] < split_dt
    df_treino = df_base.loc[mask_treino].reset_index(drop=True)

    # 1. Bias estimado exclusivamente no treino
    yhat_treino = walk_forward_tabular(
        df=df_treino,
        feature_cols=feature_cols,
        target_col=target_col,
        model_fn=model_fn,
        start_date=None,
        min_train=min_obs
    )

    if len(yhat_treino) > 0:
        y_true_treino = df_treino.loc[yhat_treino.index, target_col]
        bias_estimado = float((y_true_treino - yhat_treino).mean())
    else:
        bias_estimado = 0.0

    # 2. Walk-forward no teste oficial
    yhat_teste = walk_forward_tabular(
        df=df_base,
        feature_cols=feature_cols,
        target_col=target_col,
        model_fn=model_fn,
        start_date=split_dt,
        min_train=min_obs
    )

    if len(yhat_teste) == 0:
        return None

    y_true_teste = df_base.loc[yhat_teste.index, target_col]
    y_pred_sem_bias = yhat_teste
    y_pred_com_bias = yhat_teste + bias_estimado

    metricas_sem_bias = calcular_metricas(
        y_true_teste,
        y_pred_sem_bias,
        nome=f"{nome}"
    )

    metricas_com_bias = calcular_metricas(
        y_true_teste,
        y_pred_com_bias,
        nome=f"{nome} + bias"
    )

    resultado = metricas_com_bias.copy()
    resultado.update({
        "nome_base": nome,
        "bias_estimado": round(bias_estimado, 6),
        "metricas_sem_bias": metricas_sem_bias,
        "metricas_com_bias": metricas_com_bias,
        "y_obs": y_true_teste.tolist(),
        "y_pred_sem_bias": y_pred_sem_bias.tolist(),
        "y_pred_com_bias": y_pred_com_bias.tolist(),
        "y_pred": y_pred_com_bias.tolist(),
        "datas": [str(df_base.loc[i, "data_alvo"].date()) for i in yhat_teste.index],
    })

    return resultado

print("Funções tabulares com bias correction definidas.")


Funções tabulares com bias correction definidas.


### Observação sobre bias correction

Para manter comparabilidade com o Notebook 02, os modelos deste notebook reportam as métricas principais **com correção de bias**.

O bias é estimado exclusivamente no conjunto de treino, a partir de previsões walk-forward, pela média dos erros `observado - previsto`. Em seguida, esse valor é somado às previsões do teste.

Assim, o conjunto de teste não é usado para calibrar a correção; ele permanece reservado para avaliação final fora da amostra.

In [ ]:
# ── Helper: construir df tabular sem causar bias por dropna em sentimento ─────
def build_df_reg(base_fit, lags_inad, col_sent=None):
    """
    Monta o dataframe para walk-forward tabular.
    Se col_sent for fornecida, inclui apenas linhas onde ela não é nula.
    Se col_sent for None, usa o máximo de linhas disponíveis.
    """
    df = base_fit.copy()
    df["mes"]       = df["data"]
    df["data_alvo"] = pd.to_datetime(df["data_alvo"])
    cols = ["mes", "data_alvo"] + lags_inad + [VARIAVEL_ALVO]
    if col_sent is not None:
        cols = cols + [col_sent]
    return (
        df[cols].dropna()
        .sort_values("data_alvo")
        .reset_index(drop=True)
    )

LAGS_INAD_XGB = [f"inad_total_l{k}" for k in range(1, K_XGB + 1)]
LAGS_INAD_SVR = [f"inad_total_l{k}" for k in range(1, K_SVR + 1)]

print(f"Features base XGBoost: {LAGS_INAD_XGB}")
print(f"Features base SVR    : {LAGS_INAD_SVR}")


Helpers definidos.
Features base XGBoost: ['inad_total_l1', 'inad_total_l2', 'inad_total_l3', 'inad_total_l4', 'inad_total_l5', 'inad_total_l6']
Features base SVR    : ['inad_total_l1', 'inad_total_l2']


## 8. ARIMAX — todos os sentimentos × todos os lags

Para cada modelo de sentimento e cada lag L1–L6, estima-se um
ARIMAX(2,1,4) com aquele único lag como variável exógena.
O resultado é uma tabela de RMSE (modelo × lag).


In [10]:
print("=" * 60)
print("ARIMAX — baseline e todos os sentimentos × L1–L6")
print("=" * 60)

# Baseline ARIMA sem sentimento
print("\n  Baseline: ARIMA(2,1,4) sem sentimento...")
res_arima_base = estimar_arimax(
    nome="ARIMA(2,1,4) sem sentimento",
    base_treino=base_treino, base_teste=base_teste,
    variavel_alvo=VARIAVEL_ALVO, cols_exog=[],
    ordem=ORDEM_ARIMA,
)
if res_arima_base:
    print(f"  OK | RMSE+bias={res_arima_base['RMSE']:.4f} | bias={res_arima_base['bias_estimado']:.4f}")
else:
    print("  AVISO: baseline não estimado.")


ARIMAX — baseline e todos os sentimentos × L1–L6

  Baseline: ARIMA(2,1,4) sem sentimento...


/usr/local/python/3.12.1/lib/python3.12/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency MS will be used.
  self._init_dates(dates, freq)
/usr/local/python/3.12.1/lib/python3.12/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency MS will be used.
  self._init_dates(dates, freq)
/usr/local/python/3.12.1/lib/python3.12/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
/usr/local/python/3.12.1/lib/python3.12/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency MS will be used.
  self._init_dates(dates, freq)
/usr/local/python/3.12.1/lib/python3.12/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No fre

  OK | RMSE+bias=0.1857 | bias=0.0646


/usr/local/python/3.12.1/lib/python3.12/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


In [11]:
# Loop ARIMAX por sentimento × lag
resultados_arimax = {}

for chave, cfg in MODELOS_SENTIMENTO.items():
    resultados_arimax[chave] = {}
    print(f"\n  {cfg['label']}")
    for lag in LAGS:
        col = mapa_cols[chave][lag]
        if col is None:
            print(f"    L{lag}: coluna ausente")
            continue
        nome = f"ARIMAX + {cfg['label']} L{lag}"
        res  = estimar_arimax(
            nome=nome,
            base_treino=base_treino, base_teste=base_teste,
            variavel_alvo=VARIAVEL_ALVO,
            cols_exog=[col],
            ordem=ORDEM_ARIMA,
        )
        resultados_arimax[chave][lag] = res
        if res:
            print(f"    L{lag}: RMSE+bias={res['RMSE']:.4f}  bias={res['bias_estimado']:.4f}")
        else:
            print(f"    L{lag}: sem resultado")


/usr/local/python/3.12.1/lib/python3.12/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency MS will be used.
  self._init_dates(dates, freq)
/usr/local/python/3.12.1/lib/python3.12/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency MS will be used.
  self._init_dates(dates, freq)



  NLTK/VADER Copom


/usr/local/python/3.12.1/lib/python3.12/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
/usr/local/python/3.12.1/lib/python3.12/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency MS will be used.
  self._init_dates(dates, freq)
/usr/local/python/3.12.1/lib/python3.12/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency MS will be used.
  self._init_dates(dates, freq)
/usr/local/python/3.12.1/lib/python3.12/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
/usr/local/python/3.12.1/lib/python3.12/site-packages/statsmodels/tsa/base/tsa_model.py:473: V

    L1: RMSE+bias=0.2027  bias=0.0752


/usr/local/python/3.12.1/lib/python3.12/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
/usr/local/python/3.12.1/lib/python3.12/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency MS will be used.
  self._init_dates(dates, freq)
/usr/local/python/3.12.1/lib/python3.12/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency MS will be used.
  self._init_dates(dates, freq)
/usr/local/python/3.12.1/lib/python3.12/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
/usr/local/python/3.12.1/lib/python3.12/site-packages/statsmodels/tsa/base/tsa_model.py:473: V

...

/usr/local/python/3.12.1/lib/python3.12/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
/usr/local/python/3.12.1/lib/python3.12/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency MS will be used.
  self._init_dates(dates, freq)
/usr/local/python/3.12.1/lib/python3.12/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency MS will be used.
  self._init_dates(dates, freq)
/usr/local/python/3.12.1/lib/python3.12/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
/usr/local/python/3.12.1/lib/python3.12/site-packages/statsmodels/tsa/base/tsa_model.py:473: V

    L5: RMSE+bias=0.1934  bias=0.0559


/usr/local/python/3.12.1/lib/python3.12/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
/usr/local/python/3.12.1/lib/python3.12/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency MS will be used.
  self._init_dates(dates, freq)
/usr/local/python/3.12.1/lib/python3.12/site-packages/statsmodels/tsa/base/tsa_model.py:473: ValueWarning: No frequency information was provided, so inferred frequency MS will be used.
  self._init_dates(dates, freq)
/usr/local/python/3.12.1/lib/python3.12/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "
/usr/local/python/3.12.1/lib/python3.12/site-packages/statsmodels/tsa/base/tsa_model.py:473: V

    L6: RMSE+bias=0.1761  bias=0.0707


/usr/local/python/3.12.1/lib/python3.12/site-packages/statsmodels/base/model.py:607: ConvergenceWarning: Maximum Likelihood optimization failed to converge. Check mle_retvals
  warnings.warn("Maximum Likelihood optimization failed to "


In [12]:
# Tabela pivot RMSE — ARIMAX
linhas_arimax = []
for chave, cfg in MODELOS_SENTIMENTO.items():
    for lag in LAGS:
        res = resultados_arimax.get(chave, {}).get(lag)
        linhas_arimax.append({
            "Modelo": cfg["label"],
            f"L{lag}": round(res["RMSE"], 4) if res else np.nan,
        })

df_pivot_arimax = (
    pd.DataFrame(linhas_arimax)
    .groupby("Modelo")
    .first()
    .reset_index()
)
# Reconstruir como pivot correto
pivot_data = {}
for chave, cfg in MODELOS_SENTIMENTO.items():
    pivot_data[cfg["label"]] = {
        f"L{lag}": round(resultados_arimax[chave][lag]["RMSE"], 4)
                   if resultados_arimax[chave].get(lag) else np.nan
        for lag in LAGS
    }
df_pivot_arimax = pd.DataFrame(pivot_data).T
df_pivot_arimax.index.name = "Modelo de Sentimento"

rmse_base_local = round(res_arima_base["RMSE"], 4) if res_arima_base else np.nan
print(f"\nRMSE baseline ARIMA local com bias: {rmse_base_local}")
print(f"RMSE benchmark NB02       : {RMSE_NB02_ARIMA}")
print()
print("=== RMSE por modelo × lag — ARIMAX com bias correction ===")
display(df_pivot_arimax)

# Melhor lag por sentimento
print("\n=== Melhor lag por modelo de sentimento (ARIMAX) ===")
melhores_arimax = {}
for chave, cfg in MODELOS_SENTIMENTO.items():
    validos = {lag: resultados_arimax[chave][lag]["RMSE"]
               for lag in LAGS if resultados_arimax[chave].get(lag)}
    if not validos:
        continue
    melhor_lag  = min(validos, key=validos.get)
    melhor_rmse = validos[melhor_lag]
    melhores_arimax[chave] = {"lag": melhor_lag, "RMSE": melhor_rmse,
                               "resultado": resultados_arimax[chave][melhor_lag]}
    print(f"  {cfg['label']:25s}: L{melhor_lag}  RMSE+bias={melhor_rmse:.4f}")



RMSE baseline ARIMA local com bias: 0.1857
RMSE benchmark NB02       : 0.184052

=== RMSE por modelo × lag — ARIMAX com bias correction ===


,L1,L2,L3,L4,L5,L6
Modelo de Sentimento,,,,,,
NLTK/VADER Copom,0.2027,0.1996,0.1942,0.1922,0.2006,0.1815
Mistral Copom,0.1976,0.1856,0.1919,0.1754,0.1944,0.1866
Media dos Modelos Copom,0.1888,0.1786,0.1861,0.1926,0.1955,0.1758
FinBERT-PT-BR Estat.,0.1868,0.1856,0.1965,0.1823,0.1934,0.1761



=== Melhor lag por modelo de sentimento (ARIMAX) ===
  NLTK/VADER Copom         : L6  RMSE+bias=0.1815
  Mistral Copom            : L4  RMSE+bias=0.1754
  Media dos Modelos Copom  : L6  RMSE+bias=0.1758
  FinBERT-PT-BR Estat.     : L6  RMSE+bias=0.1761


## 9. XGBoost — todos os sentimentos × todos os lags

Base: `inad_total_l1` a `inad_total_l6` (k=6, conforme NB02).
Para cada teste, adiciona-se **um único lag** do sentimento.


In [13]:
print("=" * 60)
print("XGBoost — baseline e todos os sentimentos × L1–L6")
print("=" * 60)

def criar_xgb():
    return XGBRegressor(
        n_estimators=400, max_depth=3, learning_rate=0.05,
        subsample=0.9, colsample_bytree=0.9,
        random_state=42, objective="reg:squarederror",
    )

# Baseline XGBoost
df_xgb_base = build_df_reg(base_fit, LAGS_INAD_XGB)
print("\n  Baseline: XGBoost(k=6) sem sentimento...")
res_xgb_base = estimar_tabular(
    nome="XGBoost(k=6) sem sentimento",
    model_fn=criar_xgb,
    df_base=df_xgb_base,
    feature_cols=LAGS_INAD_XGB,
    target_col=VARIAVEL_ALVO,
    split_dt=split_dt,
    min_obs=MIN_OBS_TREINO,
)
if res_xgb_base:
    print(f"  OK | RMSE+bias={res_xgb_base['RMSE']:.4f} | bias={res_xgb_base['bias_estimado']:.4f}")


XGBoost — baseline e todos os sentimentos × L1–L6

  Baseline: XGBoost(k=6) sem sentimento...
  OK | RMSE+bias=0.3403 | bias=0.3999


In [14]:
resultados_xgb = {}

for chave, cfg in MODELOS_SENTIMENTO.items():
    resultados_xgb[chave] = {}
    print(f"\n  {cfg['label']}")
    for lag in LAGS:
        col = mapa_cols[chave][lag]
        if col is None:
            print(f"    L{lag}: coluna ausente")
            continue
        df_sent = build_df_reg(base_fit, LAGS_INAD_XGB, col_sent=col)
        feat    = LAGS_INAD_XGB + [col]
        nome    = f"XGBoost + {cfg['label']} L{lag}"
        res = estimar_tabular(
            nome=nome, model_fn=criar_xgb,
            df_base=df_sent, feature_cols=feat,
            target_col=VARIAVEL_ALVO, split_dt=split_dt,
            min_obs=MIN_OBS_TREINO,
        )
        resultados_xgb[chave][lag] = res
        if res:
            print(f"    L{lag}: RMSE+bias={res['RMSE']:.4f}  bias={res['bias_estimado']:.4f}")
        else:
            print(f"    L{lag}: sem resultado")



  NLTK/VADER Copom
    L1: RMSE+bias=0.3528  bias=0.3973
    L2: RMSE+bias=0.3407  bias=0.4082
    L3: RMSE+bias=0.3300  bias=0.4074
    L4: RMSE+bias=0.3699  bias=0.4038
    L5: RMSE+bias=0.3483  bias=0.3979
    L6: RMSE+bias=0.3730  bias=0.3997

  Mistral Copom
    L1: RMSE+bias=0.2924  bias=0.4232
    L2: RMSE+bias=0.2800  bias=0.4055
    L3: RMSE+bias=0.3234  bias=0.3961
    L4: RMSE+bias=0.3328  bias=0.3958
    L5: RMSE+bias=0.3214  bias=0.3757
    L6: RMSE+bias=0.3272  bias=0.3717

  Media dos Modelos Copom
    L1: RMSE+bias=0.2869  bias=0.4329
    L2: RMSE+bias=0.3042  bias=0.4317
    L3: RMSE+bias=0.3310  bias=0.4228
    L4: RMSE+bias=0.3368  bias=0.4286
    L5: RMSE+bias=0.3123  bias=0.3808
    L6: RMSE+bias=0.3008  bias=0.4155

  FinBERT-PT-BR Estat.
    L1: RMSE+bias=0.3350  bias=0.4034
    L2: RMSE+bias=0.3432  bias=0.4117
    L3: RMSE+bias=0.3336  bias=0.3988
    L4: RMSE+bias=0.2798  bias=0.3824
    L5: RMSE+bias=0.3181  bias=0.3903
    L6: RMSE+bias=0.3525  bias=0.4193


In [15]:
# Tabela pivot RMSE — XGBoost
pivot_xgb = {}
for chave, cfg in MODELOS_SENTIMENTO.items():
    pivot_xgb[cfg["label"]] = {
        f"L{lag}": round(resultados_xgb[chave][lag]["RMSE"], 4)
                   if resultados_xgb[chave].get(lag) else np.nan
        for lag in LAGS
    }
df_pivot_xgb = pd.DataFrame(pivot_xgb).T
df_pivot_xgb.index.name = "Modelo de Sentimento"

print(f"RMSE baseline XGBoost com bias: {round(res_xgb_base['RMSE'], 4) if res_xgb_base else 'N/A'}")
print()
print("=== RMSE por modelo × lag — XGBoost com bias correction ===")
display(df_pivot_xgb)

print("\n=== Melhor lag por modelo de sentimento (XGBoost) ===")
melhores_xgb = {}
for chave, cfg in MODELOS_SENTIMENTO.items():
    validos = {lag: resultados_xgb[chave][lag]["RMSE"]
               for lag in LAGS if resultados_xgb[chave].get(lag)}
    if not validos:
        continue
    melhor_lag  = min(validos, key=validos.get)
    melhor_rmse = validos[melhor_lag]
    melhores_xgb[chave] = {"lag": melhor_lag, "RMSE": melhor_rmse,
                            "resultado": resultados_xgb[chave][melhor_lag]}
    print(f"  {cfg['label']:25s}: L{melhor_lag}  RMSE+bias={melhor_rmse:.4f}")


RMSE baseline XGBoost com bias: 0.3403

=== RMSE por modelo × lag — XGBoost com bias correction ===


,L1,L2,L3,L4,L5,L6
Modelo de Sentimento,,,,,,
NLTK/VADER Copom,0.3528,0.3407,0.3300,0.3699,0.3483,0.3730
Mistral Copom,0.2924,0.2800,0.3234,0.3328,0.3214,0.3272
Media dos Modelos Copom,0.2869,0.3042,0.3310,0.3368,0.3123,0.3008
FinBERT-PT-BR Estat.,0.3350,0.3432,0.3336,0.2798,0.3181,0.3525



=== Melhor lag por modelo de sentimento (XGBoost) ===
  NLTK/VADER Copom         : L3  RMSE+bias=0.3300
  Mistral Copom            : L2  RMSE+bias=0.2800
  Media dos Modelos Copom  : L1  RMSE+bias=0.2869
  FinBERT-PT-BR Estat.     : L4  RMSE+bias=0.2798


## 10. SVR — todos os sentimentos × todos os lags

Base: `inad_total_l1` e `inad_total_l2` (k=2, conforme NB02).
StandardScaler aplicado internamente pelo Pipeline.


In [16]:
print("=" * 60)
print("SVR — baseline e todos os sentimentos × L1–L6")
print("=" * 60)

def criar_svr():
    return Pipeline([
        ("scaler", StandardScaler()),
        ("svr", SVR(kernel="rbf", C=10.0, epsilon=0.05)),
    ])

# Baseline SVR
df_svr_base = build_df_reg(base_fit, LAGS_INAD_SVR)
print("\n  Baseline: SVR(k=2) sem sentimento...")
res_svr_base = estimar_tabular(
    nome="SVR(k=2) sem sentimento",
    model_fn=criar_svr,
    df_base=df_svr_base,
    feature_cols=LAGS_INAD_SVR,
    target_col=VARIAVEL_ALVO,
    split_dt=split_dt,
    min_obs=MIN_OBS_TREINO,
)
if res_svr_base:
    print(f"  OK | RMSE+bias={res_svr_base['RMSE']:.4f} | bias={res_svr_base['bias_estimado']:.4f}")


SVR — baseline e todos os sentimentos × L1–L6

  Baseline: SVR(k=2) sem sentimento...
  OK | RMSE+bias=0.3083 | bias=0.3631


In [17]:
resultados_svr = {}

for chave, cfg in MODELOS_SENTIMENTO.items():
    resultados_svr[chave] = {}
    print(f"\n  {cfg['label']}")
    for lag in LAGS:
        col = mapa_cols[chave][lag]
        if col is None:
            print(f"    L{lag}: coluna ausente")
            continue
        df_sent = build_df_reg(base_fit, LAGS_INAD_SVR, col_sent=col)
        feat    = LAGS_INAD_SVR + [col]
        nome    = f"SVR + {cfg['label']} L{lag}"
        res = estimar_tabular(
            nome=nome, model_fn=criar_svr,
            df_base=df_sent, feature_cols=feat,
            target_col=VARIAVEL_ALVO, split_dt=split_dt,
            min_obs=MIN_OBS_TREINO,
        )
        resultados_svr[chave][lag] = res
        if res:
            print(f"    L{lag}: RMSE+bias={res['RMSE']:.4f}  bias={res['bias_estimado']:.4f}")
        else:
            print(f"    L{lag}: sem resultado")



  NLTK/VADER Copom
    L1: RMSE+bias=0.4456  bias=0.3114
    L2: RMSE+bias=0.3803  bias=0.3061
    L3: RMSE+bias=0.3906  bias=0.3802
    L4: RMSE+bias=0.4074  bias=0.2707
    L5: RMSE+bias=0.3745  bias=0.3910
    L6: RMSE+bias=0.5386  bias=0.4386

  Mistral Copom
    L1: RMSE+bias=0.4179  bias=0.4259
    L2: RMSE+bias=0.4075  bias=0.4577
    L3: RMSE+bias=0.3608  bias=0.4190
    L4: RMSE+bias=0.3326  bias=0.3857
    L5: RMSE+bias=0.3480  bias=0.3557
    L6: RMSE+bias=0.3661  bias=0.3971

  Media dos Modelos Copom
    L1: RMSE+bias=0.4224  bias=0.4305
    L2: RMSE+bias=0.4533  bias=0.4954
    L3: RMSE+bias=0.3706  bias=0.3846
    L4: RMSE+bias=0.3039  bias=0.4095
    L5: RMSE+bias=0.3812  bias=0.3389
    L6: RMSE+bias=0.3428  bias=0.4054

  FinBERT-PT-BR Estat.
    L1: RMSE+bias=0.3827  bias=0.3162
    L2: RMSE+bias=0.3447  bias=0.3573
    L3: RMSE+bias=0.3358  bias=0.4102
    L4: RMSE+bias=0.3162  bias=0.3435
    L5: RMSE+bias=0.3102  bias=0.3729
    L6: RMSE+bias=0.4953  bias=0.4645


In [18]:
# Tabela pivot RMSE — SVR
pivot_svr = {}
for chave, cfg in MODELOS_SENTIMENTO.items():
    pivot_svr[cfg["label"]] = {
        f"L{lag}": round(resultados_svr[chave][lag]["RMSE"], 4)
                   if resultados_svr[chave].get(lag) else np.nan
        for lag in LAGS
    }
df_pivot_svr = pd.DataFrame(pivot_svr).T
df_pivot_svr.index.name = "Modelo de Sentimento"

print(f"RMSE baseline SVR com bias: {round(res_svr_base['RMSE'], 4) if res_svr_base else 'N/A'}")
print()
print("=== RMSE por modelo × lag — SVR com bias correction ===")
display(df_pivot_svr)

print("\n=== Melhor lag por modelo de sentimento (SVR) ===")
melhores_svr = {}
for chave, cfg in MODELOS_SENTIMENTO.items():
    validos = {lag: resultados_svr[chave][lag]["RMSE"]
               for lag in LAGS if resultados_svr[chave].get(lag)}
    if not validos:
        continue
    melhor_lag  = min(validos, key=validos.get)
    melhor_rmse = validos[melhor_lag]
    melhores_svr[chave] = {"lag": melhor_lag, "RMSE": melhor_rmse,
                            "resultado": resultados_svr[chave][melhor_lag]}
    print(f"  {cfg['label']:25s}: L{melhor_lag}  RMSE+bias={melhor_rmse:.4f}")


RMSE baseline SVR com bias: 0.3083

=== RMSE por modelo × lag — SVR com bias correction ===


,L1,L2,L3,L4,L5,L6
Modelo de Sentimento,,,,,,
NLTK/VADER Copom,0.4456,0.3803,0.3906,0.4074,0.3745,0.5386
Mistral Copom,0.4179,0.4075,0.3608,0.3326,0.3480,0.3661
Media dos Modelos Copom,0.4224,0.4533,0.3706,0.3039,0.3812,0.3428
FinBERT-PT-BR Estat.,0.3827,0.3447,0.3358,0.3162,0.3102,0.4953



=== Melhor lag por modelo de sentimento (SVR) ===
  NLTK/VADER Copom         : L5  RMSE+bias=0.3745
  Mistral Copom            : L4  RMSE+bias=0.3326
  Media dos Modelos Copom  : L4  RMSE+bias=0.3039
  FinBERT-PT-BR Estat.     : L5  RMSE+bias=0.3102


## 11. Resumo e exportação para o Notebook 07

In [19]:
# Resumo: melhor sentimento por algoritmo
print("=== Melhor sentimento por algoritmo ===")

def melhor_de(melhores_dict):
    validos = {k: v for k, v in melhores_dict.items() if v}
    if not validos:
        return None, None, None
    melhor_chave = min(validos, key=lambda k: validos[k]["RMSE"])
    return melhor_chave, validos[melhor_chave]["lag"], validos[melhor_chave]["RMSE"]

chave_a, lag_a, rmse_a = melhor_de(melhores_arimax)
chave_x, lag_x, rmse_x = melhor_de(melhores_xgb)
chave_s, lag_s, rmse_s = melhor_de(melhores_svr)

for algo, chave, lag, rmse in [
    ("ARIMAX",   chave_a, lag_a, rmse_a),
    ("XGBoost",  chave_x, lag_x, rmse_x),
    ("SVR",      chave_s, lag_s, rmse_s),
]:
    label = MODELOS_SENTIMENTO[chave]["label"] if chave else "N/A"
    print(f"  {algo:<10}: {label}  L{lag}  RMSE+bias={rmse:.4f}")


=== Melhor sentimento por algoritmo ===
  ARIMAX    : Mistral Copom  L4  RMSE+bias=0.1754
  XGBoost   : FinBERT-PT-BR Estat.  L4  RMSE+bias=0.2798
  SVR       : Media dos Modelos Copom  L4  RMSE+bias=0.3039


## 12. Comparativo final com bias correction

A tabela abaixo consolida os modelos estimados no Notebook 06 usando as métricas com correção de bias, para manter o mesmo critério de comparação adotado no Notebook 02.

In [20]:
# =============================================================
# Comparativo final — métricas com bias correction
# =============================================================

def linha_resultado(resultado, grupo, algoritmo, usa_sentimento, sentimento="", lag=""):
    if resultado is None:
        return None
    return {
        "Modelo": resultado.get("Modelo", resultado.get("nome_base", "")),
        "Grupo": grupo,
        "Algoritmo": algoritmo,
        "Usa_sentimento": usa_sentimento,
        "Sentimento": sentimento,
        "Lag_sentimento": lag,
        "MAE": resultado.get("MAE", np.nan),
        "RMSE": resultado.get("RMSE", np.nan),
        "R2": resultado.get("R2", np.nan),
        "Bias": resultado.get("Bias", np.nan),
        "Slope": resultado.get("Slope", np.nan),
        "Intercept": resultado.get("Intercept", np.nan),
        "bias_estimado_treino": resultado.get("bias_estimado", np.nan),
        "N": resultado.get("N", np.nan),
    }

linhas = []

# Baselines locais sem sentimento
linhas.append(linha_resultado(res_arima_base, "Sem sentimento", "ARIMA", False))
linhas.append(linha_resultado(res_xgb_base,   "Sem sentimento", "XGBoost", False))
linhas.append(linha_resultado(res_svr_base,   "Sem sentimento", "SVR", False))

# Melhores modelos com sentimento por algoritmo
for chave, item in melhores_arimax.items():
    linhas.append(linha_resultado(
        item["resultado"], "Com sentimento", "ARIMAX", True,
        sentimento=MODELOS_SENTIMENTO[chave]["label"], lag=f"L{item['lag']}"
    ))

for chave, item in melhores_xgb.items():
    linhas.append(linha_resultado(
        item["resultado"], "Com sentimento", "XGBoost", True,
        sentimento=MODELOS_SENTIMENTO[chave]["label"], lag=f"L{item['lag']}"
    ))

for chave, item in melhores_svr.items():
    linhas.append(linha_resultado(
        item["resultado"], "Com sentimento", "SVR", True,
        sentimento=MODELOS_SENTIMENTO[chave]["label"], lag=f"L{item['lag']}"
    ))

comparativo_final_nb06 = (
    pd.DataFrame([l for l in linhas if l is not None])
    .sort_values("RMSE")
    .reset_index(drop=True)
)

comparativo_final_nb06["Ranking_RMSE"] = comparativo_final_nb06.index + 1

print("=== Comparativo final Notebook 06 — métricas com bias correction ===")
display(comparativo_final_nb06.round(6))

# Exportação para o Notebook 07
comparativo_final_nb06.to_csv("resultados_notebook06.csv", index=False)


=== Comparativo final Notebook 06 — métricas com bias correction ===


,Modelo,Grupo,Algoritmo,Usa_sentimento,Sentimento,Lag_sentimento,MAE,RMSE,R2,Bias,Slope,Intercept,bias_estimado_treino,N,Ranking_RMSE
0,ARIMAX + Mistral Copom L4 + bias,Com sentimento,ARIMAX,True,Mistral Copom,L4,0.132543,0.175374,0.716628,-0.012423,0.785812,0.746819,0.074078,24,1
1,ARIMAX + Media dos Modelos Copom L6 + bias,Com sentimento,ARIMAX,True,Media dos Modelos Copom,L6,0.135281,0.175807,0.715227,-0.008185,0.804033,0.680107,0.071777,24,2
2,ARIMAX + FinBERT-PT-BR Estat. L6 + bias,Com sentimento,ARIMAX,True,FinBERT-PT-BR Estat.,L6,0.144287,0.176063,0.714397,-0.001782,0.774955,0.773404,0.070680,24,3
3,ARIMAX + NLTK/VADER Copom L6 + bias,Com sentimento,ARIMAX,True,NLTK/VADER Copom,L6,0.143467,0.181516,0.696433,0.002214,0.809732,0.650167,0.081420,24,4
4,"ARIMA(2,1,4) sem sentimento + bias",Sem sentimento,ARIMA,False,,,0.145419,0.185725,0.682190,0.006215,0.746013,0.864643,0.064559,24,5
5,XGBoost + FinBERT-PT-BR Estat. L4 + bias,Com sentimento,XGBoost,True,FinBERT-PT-BR Estat.,L4,0.245829,0.279762,0.278886,-0.117465,0.317712,2.456861,0.382433,24,6
6,XGBoost + Mistral Copom L2 + bias,Com sentimento,XGBoost,True,Mistral Copom,L2,0.211347,0.280026,0.277525,-0.161774,0.596299,1.545964,0.405546,24,7
7,XGBoost + Media dos Modelos Copom L1 + bias,Com sentimento,XGBoost,True,Media dos Modelos Copom,L1,0.234909,0.286945,0.241380,-0.162040,0.412791,2.175434,0.432902,24,8
8,SVR + Media dos Modelos Copom L4 + bias,Com sentimento,SVR,True,Media dos Modelos Copom,L4,0.240237,0.303947,0.148818,-0.099941,0.648969,1.303539,0.409485,24,9
9,SVR(k=2) sem sentimento + bias,Sem sentimento,SVR,False,,,0.273558,0.308317,0.124166,-0.145103,0.382565,2.262132,0.363110,24,10
